# Multi-Head Attention & Positional Encoding

**Companion lesson:** https://ml-viz.vercel.app/courses/transformers/02-multi-head-and-positional

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Multi-head attention, from scratch

Split $d_{model}$ across $h$ heads, attend independently, concatenate, and mix with $W_O$. The reshape into heads is the only subtle part.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads):
    T, d_model = X.shape
    d_k = d_model // n_heads
    Q = (X @ Wq).reshape(T, n_heads, d_k).transpose(1, 0, 2)   # (h, T, d_k)
    K = (X @ Wk).reshape(T, n_heads, d_k).transpose(1, 0, 2)
    V = (X @ Wv).reshape(T, n_heads, d_k).transpose(1, 0, 2)
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)          # (h, T, T)
    W = softmax(scores, axis=-1)
    ctx = W @ V                                               # (h, T, d_k)
    ctx = ctx.transpose(1, 0, 2).reshape(T, d_model)         # concat heads
    return ctx @ Wo, W

d_model, h, T = 64, 8, 6
rng = np.random.RandomState(0)
X = rng.randn(T, d_model)
Wq, Wk, Wv, Wo = (rng.randn(d_model, d_model)*0.1 for _ in range(4))
out, W = multi_head_attention(X, Wq, Wk, Wv, Wo, h)
print('d_k per head =', d_model//h, '| output:', out.shape, '| per-head weights:', W.shape)

## Different heads learn different patterns

Even with random weights, each head produces a distinct attention map — capacity the model uses to track syntax, coreference, etc. simultaneously.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(W[i], cmap='viridis'); ax.set_title(f'head {i}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Eight attention heads, eight views of the same sequence'); plt.show()

## Sinusoidal positional encoding

Attention is permutation-invariant, so we add a position signal. Each dimension is a sinusoid of a different wavelength — low dims = coarse position, high dims = fine.

In [ ]:
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]; i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2*(i//2))/d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2]); pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

pe = positional_encoding(50, 64)
plt.imshow(pe, cmap='RdBu', aspect='auto')
plt.xlabel('embedding dimension'); plt.ylabel('position'); plt.title('Positional encoding')
plt.colorbar(); plt.show()
# nearby positions have similar encodings — show the dot-product structure
sim = pe @ pe.T
print('PE similarity is highest on the diagonal (nearby positions):', bool(sim[10,10] >= sim[10].max()-1e-9))

## Key takeaways

- Multi-head attention runs `h` attention views in parallel, each of width `d_model/h`.
- Concatenating heads + a `W_O` projection mixes them back to `d_model`.
- Self-attention is permutation-invariant — **positional encoding** injects order.
- Sinusoidal encodings give every position a unique, smoothly-varying signature.